# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhruvaiyer2-netizen/FlyRankMLProject/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb huggingface_hub scikit-learn

from huggingface_hub import login
login()  # paste your HF token

import duckdb, numpy as np, pandas as pd
from huggingface_hub import HfFileSystem
duckdb.register_filesystem(HfFileSystem())

FACT = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
DIM  = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'

# One row per page per month (ratio-of-sums for ctr and position); Feb is for the time-aware design
monthly = duckdb.sql(f"""
    SELECT month, content_hash_id, client_hash_id,
      SUM(gsc_impressions) AS impressions,
      SUM(gsc_clicks) AS clicks,
      SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
      100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
      MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM '{FACT}'
    WHERE month IN ('2026-02', '2026-03', '2026-04')
    GROUP BY month, content_hash_id, client_hash_id
""").df()

dim = duckdb.sql(f"""
    SELECT content_hash_id, word_count, main_intent, content_created_date, is_deleted
    FROM '{DIM}'
""").df()

MIN_IMP = 500
TIER_BINS = [0, 3, 10, 20, float("inf")]
TIER_LABELS = ["1-3", "4-10", "11-20", "21+"]
INTENTS = ["informational", "commercial", "transactional", "navigational"]

def prep(df):
    """Volume filter, position tier, tier-relative CTR gap (the Week 4 rule's score)."""
    df = df[df["impressions"] >= MIN_IMP].copy()
    df["tier"] = pd.cut(df["avg_position"], TIER_BINS, labels=TIER_LABELS)
    df["tier_median_ctr"] = df.groupby("tier", observed=True)["ctr"].transform("median")
    df["ctr_gap_score"] = df["tier_median_ctr"] - df["ctr"]
    return df

def build(feat_month, label_month, drop_deleted=True):
    """Features from feat_month only; label = underperforming its tier in label_month."""
    feat = prep(monthly[monthly["month"] == feat_month])
    lab = prep(monthly[monthly["month"] == label_month])[["content_hash_id", "client_hash_id", "ctr_gap_score"]]
    lab = lab.rename(columns={"ctr_gap_score": "label_gap"})

    d = feat.merge(lab, on=["content_hash_id", "client_hash_id"], how="inner")
    d["y"] = (d["label_gap"] > 0).astype(int)
    d = d.merge(dim, on="content_hash_id", how="left")
    if drop_deleted:
        d = d[d["is_deleted"] != True]
    d = d.copy()

    ref = pd.Timestamp(feat_month + "-01") + pd.offsets.MonthEnd(0)   # decision moment = end of feature month
    d["word_count_missing"] = d["word_count"].isna().astype(int)
    d["word_count_filled"] = d["word_count"].fillna(-1)
    d["content_age_days"] = (ref - pd.to_datetime(d["content_created_date"])).dt.days.clip(lower=0).fillna(-1)
    for it in INTENTS:
        d[f"intent_{it}"] = (d["main_intent"] == it).astype(int)
    return d.reset_index(drop=True)

FEATURES = ["ctr", "avg_position", "impressions", "ctr_gap_score",
            "word_count_filled", "word_count_missing", "content_age_days"] + [f"intent_{i}" for i in INTENTS]

mar_apr = build("2026-03", "2026-04")   # features: March, label: April
feb_mar = build("2026-02", "2026-03")   # features: Feb,   label: March
print(f"Mar→Apr: {len(mar_apr):,} pages, base rate {mar_apr['y'].mean():.3f}")
print(f"Feb→Mar: {len(feb_mar):,} pages, base rate {feb_mar['y'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Mar→Apr: 51,494 pages, base rate 0.513
Feb→Mar: 43,797 pages, base rate 0.496


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #1, The Anatomy of Growing Content (growing pages average ~3.2K words vs ~2.3K for declining, and are younger: 184 vs 230 days). The paper reports its sample sizes (74,187 rising vs 45,272 falling) and labels the result observational, which helps readers calibrate. Two questions I would ask:

Label origin. "Growing" is defined as a >10% change in impressions between the last 30 days and the prior 30. A short-window change like this can depend on the base period, since a page with a low prior-30 base can look like it is growing. Would the rising/falling split hold if both windows needed a minimum impression count?
Does the comparison support the action? The appendix reports that word count and content age correlate at r ≈ −0.52, so longer pages may partly be newer-vintage pages. Does the length gap persist within age bands, as the paper already does for the model comparison in Finding #10? A tracked before/after on expanded pages vs matched unexpanded pages (the paper's own "Measure" step) would support the "expand thin pages" action.

Finding #4, The Freshness Multiplier (365+ day pages refreshed within 30 days: health 10.7 → 34.5, impressions 71 → 4,039). I appreciate that the paper itself demotes the 361+ freshness bucket (283:1 on one declining page) and separates 181-360 from it. Two questions I would ask:

Validation design. Is the 365+ comparison between different pages (refreshed vs not) at one point in time, or the same pages before and after? If it is the former, pages chosen for refresh may already differ, for example by showing demand, so the gap may not isolate the refresh. A matched before/after comparison with untouched controls would better support "one of the strongest levers."
Construction. Health score includes impressions (30 points), so "3.2x health" and "57x impressions" are not fully independent confirmations. The active subset also requires impressions > 0 and sessions > 0, a survivor filter the paper flags for the 361+ cell. Would the same caution apply here? Group sizes and an interval for the 71 → 4,039 gap would help readers judge stability.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**What I did.** I re-ran the Week 5 model under three designs, using 5 random seeds each. In every design the features come from the earlier month, and the label (does the page sit below its position-tier median CTR the next month?) comes from the later month.

- **Before:** a random split of pages. My Week 5 split was already grouped by client, so this is the naive version, shown to measure what it would have overstated. Pages from the same client can land in both train and test.
- **After 1:** a split grouped by client, so test pages come from clients the model never saw in training.
- **After 2:** a time-aware version. The model trains on February→March data and is tested on March→April data for clients held out of training.

**What I measured (averages over 5 seeds, ROC-AUC).**

| Design | Week 4 rule | Random Forest | Gain range across seeds |
|---|---|---|---|
| Random page split | 0.824 | 0.845 | +0.021 to +0.022 |
| Client-grouped | 0.820 | 0.829 | +0.003 to +0.016 |
| Forward in time, unseen clients | 0.825 | 0.831 | -0.001 to +0.011 |

The forest's measured advantage shrank from about +0.021 under the random split to about +0.006 to +0.009 under the stricter designs, and in the forward-in-time design it was not positive in every seed. The random split was therefore optimistic for this question. Logistic Regression scored 0.815 to 0.825, roughly level with or below the rule.

At the top of the queue the forest still led. Precision among the top 500 pages was 0.939 (client-grouped) and 0.934 (forward in time) vs about 0.875 for the rule, compared with 0.976 vs 0.882 under the random split. Test sets had about 10 to 11 clients under the stricter designs, so these numbers rest on a small number of clients.

**Where the model is wrong (real examples, client-grouped split).** Two patterns showed up:
- *Confident false positives:* pages with about 1,400 to 8,400 March impressions and a March CTR under 0.1% (the tier median is about 0.2%). The model expected them to keep underperforming, but in April they recovered to above their tier median.
- *Confident false negatives:* pages with March CTR of about 0.9% to 2.6%, well above the tier median. The model expected them to stay healthy, but in April they fell below the median. Most missed the cutoff only slightly (6 of the 8 examples by less than 0.06 CTR points), and a couple had only 700 to 850 impressions, where a few clicks can swing CTR a lot.

Both patterns look like month-to-month movement back toward the middle, which a model trained on one month cannot see coming. Error rates by size of the March gap were 35%, 32%, 20% and 10% from the closest-to-median band to the farthest. Pages near the tier median are hard to call because a small change in CTR moves them across the cutoff.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit

def fit_eval(train, test, label, n_est=300, k=500):
    X_tr, y_tr = train[FEATURES], train["y"].values
    X_te, y_te = test[FEATURES], test["y"].values
    rf = RandomForestClassifier(n_estimators=n_est, min_samples_leaf=20, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    lg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
    p_rf, p_lg = rf.predict_proba(X_te)[:, 1], lg.predict_proba(X_te)[:, 1]
    rule = test["ctr_gap_score"].values
    top_rule = np.lexsort((-test["impressions"].values, -rule))[:k]   # Week 4 rule, ties broken by impressions
    top_rf = np.argsort(-p_rf)[:k]
    out = {"design": label, "n_train": len(train), "n_test": len(test),
           "test_clients": test["client_hash_id"].nunique(), "test_base_rate": y_te.mean(),
           "rule_auc": roc_auc_score(y_te, rule), "logit_auc": roc_auc_score(y_te, p_lg),
           "rf_auc": roc_auc_score(y_te, p_rf),
           "rule_p@500": y_te[top_rule].mean(), "rf_p@500": y_te[top_rf].mean()}
    return out, test.assign(p_rf=p_rf)

def grouped_split(df, seed):
    tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed).split(df, groups=df["client_hash_id"]))
    return df.iloc[tr_i], df.iloc[te_i]

def split_random(seed):
    return train_test_split(mar_apr, test_size=0.3, random_state=seed)

def split_grouped(seed):
    return grouped_split(mar_apr, seed)

def split_forward_unseen(seed):
    rng = np.random.default_rng(seed)
    clients = mar_apr["client_hash_id"].unique()
    held = set(rng.choice(clients, size=max(1, int(0.3 * len(clients))), replace=False))
    return feb_mar[~feb_mar["client_hash_id"].isin(held)], mar_apr[mar_apr["client_hash_id"].isin(held)]

designs = {"BEFORE: random page split": split_random,
           "AFTER 1: client-grouped": split_grouped,
           "AFTER 2: train Feb→Mar, test Mar→Apr, unseen clients": split_forward_unseen}

rows = []
for name, fn in designs.items():
    for seed in range(5):
        tr, te = fn(seed)
        r, _ = fit_eval(tr, te, name, n_est=150)
        r["seed"] = seed
        rows.append(r)
runs = pd.DataFrame(rows)
runs["rf_minus_rule_auc"] = runs["rf_auc"] - runs["rule_auc"]

summary = runs.groupby("design", sort=False).agg(
    n_test=("n_test", "mean"), clients=("test_clients", "mean"),
    rule_auc=("rule_auc", "mean"), logit_auc=("logit_auc", "mean"), rf_auc=("rf_auc", "mean"),
    gain_min=("rf_minus_rule_auc", "min"), gain_max=("rf_minus_rule_auc", "max"),
    rule_p500=("rule_p@500", "mean"), rf_p500=("rf_p@500", "mean")).round(3)
print(summary)
tr_g, te_g = split_grouped(0)
res_g, scored = fit_eval(tr_g, te_g, "grouped (seed 0)")
scored["pred"] = (scored["p_rf"] >= 0.5).astype(int)
scored["error"] = (scored["pred"] != scored["y"]).astype(int)
cols = ["content_hash_id", "tier", "avg_position", "impressions", "ctr", "tier_median_ctr", "p_rf", "label_gap"]

print("Confident false positives (predicted to keep underperforming; actually recovered in April):")
print(scored[(scored.pred == 1) & (scored.y == 0)].sort_values("p_rf", ascending=False)[cols].head(8).round(3).to_string(index=False))
print("\nConfident false negatives (predicted to recover; actually still underperformed):")
print(scored[(scored.pred == 0) & (scored.y == 1)].sort_values("p_rf")[cols].head(8).round(3).to_string(index=False))

# Do errors concentrate where March CTR was already close to its tier median?
scored["gap_band"] = pd.qcut(scored["ctr_gap_score"].abs(), 4, duplicates="drop")
print("\nError rate by |March gap| band (first band = closest to the tier median):")
print(scored.groupby("gap_band", observed=True)["error"].agg(error_rate="mean", n="size").round(3))

                                                     n_test  clients  \
design                                                                 
BEFORE: random page split                           15449.0     30.2   
AFTER 1: client-grouped                             13204.8     11.0   
AFTER 2: train Feb→Mar, test Mar→Apr, unseen cl...  15610.0     10.0   

                                                    rule_auc  logit_auc  \
design                                                                    
BEFORE: random page split                              0.824      0.825   
AFTER 1: client-grouped                                0.820      0.815   
AFTER 2: train Feb→Mar, test Mar→Apr, unseen cl...     0.825      0.820   

                                                    rf_auc  gain_min  \
design                                                                 
BEFORE: random page split                            0.845     0.021   
AFTER 1: client-grouped                        

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Features used:** `ctr`, `avg_position`, `impressions` and `ctr_gap_score` (all from the feature month only), `word_count`, a word-count-missing flag, `content_age_days` (measured at the end of the feature month), and four intent flags.

**Checks I ran and what I measured:**
1. **Column check.** No label columns, no FlyRank product flags (`health_score`, `priority_score`, `action_type`, `needs_ctr_fix`, `is_quick_win`, `refresh_tier`), and no optimized/updated date fields are in the feature set.
2. **Window check.** Features come from one month and labels from the next (Feb 1–28, Mar 1–31, Apr 1–30). The windows do not overlap.
3. **Single-feature check.** The strongest single features were `ctr_gap_score` (AUC 0.828) and `ctr` (0.823). Every other feature sat between about 0.50 and 0.54. No single column comes close to 1.0, so none appears to give away the label. It also shows that nearly all the signal is the same one the Week 4 rule already uses, which fits the forest only modestly beating the rule.
4. **Shuffled-label control.** Training on shuffled labels gave AUC 0.475, about 0.5 as expected. The pipeline is not producing signal from nowhere.
5. **Tier-median check.** The rule's AUC was 0.8283 with tier medians computed from all March pages and 0.8276 with medians from training pages only. The difference (0.0007) is negligible.
6. **Row filters.** Of 61,924 March pages with at least 500 impressions, 51,496 (83.2%) also had at least 500 in April and so received a label. Dropping pages deleted as of today removed only 2 more. The model therefore only speaks to pages that stay visible, and 16.8% of visible March pages fall outside its scope.

**Result:** I found no evidence of leakage in these checks.

**Known limits:** `dim_content` is a current snapshot, not an as-of-March one, so `word_count` and intent could differ from what was true in March. `word_count` is missing for roughly a quarter of live pages (about 26% in the earlier check), which is why the missing flag is included. The test also rests on one or two month-to-month transitions and a small number of large clients.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
PRODUCT_FLAGS = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win",
                 "refresh_tier", "last_optimized_date", "content_updated_date", "optimization_eligible_date"]

# 1. Column hygiene
print("Label columns in FEATURES:", [f for f in FEATURES if f in ("y", "label_gap")] or "none")
print("Product flags / post-decision date fields in FEATURES:", [f for f in FEATURES if f in PRODUCT_FLAGS] or "none")

# 2. Window check: feature month and label month don't overlap
print("\nDate range actually present per month:")
print(monthly.groupby("month").agg(first=("first_date", "min"), last=("last_date", "max")))

# 3. Single-feature AUC on held-out clients (anything near 1.0 would be suspicious)
single = {f: max(roc_auc_score(te_g["y"], te_g[f]), 1 - roc_auc_score(te_g["y"], te_g[f])) for f in FEATURES}
print("\nSingle-feature AUC (held-out clients):")
print(pd.Series(single).sort_values(ascending=False).round(3))

# 4. Shuffled-label control: AUC should be ~0.5
y_shuf = np.random.default_rng(0).permutation(tr_g["y"].values)
rf_shuf = RandomForestClassifier(n_estimators=150, min_samples_leaf=20, random_state=42, n_jobs=-1).fit(tr_g[FEATURES], y_shuf)
print("\nAUC with shuffled training labels (expect ~0.5):",
      round(roc_auc_score(te_g["y"], rf_shuf.predict_proba(te_g[FEATURES])[:, 1]), 3))

# 5. Tier medians from all March pages vs from training pages only
train_med = {str(k): v for k, v in tr_g.groupby("tier", observed=True)["ctr"].median().items()}
alt_gap = te_g["tier"].astype(str).map(train_med).astype(float) - te_g["ctr"]
print("\nRule AUC, tier medians from all March pages:   ", round(roc_auc_score(te_g["y"], te_g["ctr_gap_score"]), 4))
print("Rule AUC, tier medians from training pages only:", round(roc_auc_score(te_g["y"], alt_gap), 4))

# 6. Row filters that use later information
visible_mar = prep(monthly[monthly["month"] == "2026-03"])
with_deleted = build("2026-03", "2026-04", drop_deleted=False)
print(f"\nMarch pages with >=500 impressions:            {len(visible_mar):,}")
print(f"...also >=500 in April (i.e. get a label):      {len(with_deleted):,} ({len(with_deleted)/len(visible_mar):.1%})")
print(f"...after dropping pages deleted as of today:    {len(mar_apr):,} (dropped {len(with_deleted)-len(mar_apr):,})")

held = set(te_g["client_hash_id"])   # same held-out clients, so the comparison is controlled
r_keep, _ = fit_eval(with_deleted[~with_deleted["client_hash_id"].isin(held)],
                     with_deleted[with_deleted["client_hash_id"].isin(held)], "keep deleted pages")
pd.DataFrame([res_g, r_keep]).set_index("design")[["n_test", "rule_auc", "rf_auc", "rule_p@500", "rf_p@500"]].round(3)

Label columns in FEATURES: none
Product flags / post-decision date fields in FEATURES: none

Date range actually present per month:
             first       last
month                        
2026-02 2026-02-01 2026-02-28
2026-03 2026-03-01 2026-03-31
2026-04 2026-04-01 2026-04-30

Single-feature AUC (held-out clients):
ctr_gap_score           0.828
ctr                     0.823
avg_position            0.541
intent_informational    0.521
content_age_days        0.521
word_count_filled       0.516
intent_transactional    0.515
impressions             0.515
word_count_missing      0.513
intent_commercial       0.506
intent_navigational     0.501
dtype: float64

AUC with shuffled training labels (expect ~0.5): 0.475

Rule AUC, tier medians from all March pages:    0.8283
Rule AUC, tier medians from training pages only: 0.8276

March pages with >=500 impressions:            61,924
...also >=500 in April (i.e. get a label):      51,496 (83.2%)
...after dropping pages deleted as of today:   

,n_test,rule_auc,rf_auc,rule_p@500,rf_p@500
design,,,,,
grouped (seed 0),4477,0.828,0.836,0.852,0.87
keep deleted pages,4479,0.828,0.837,0.852,0.87


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence (before):**
"The Random Forest beats the Week 4 rule and works on new clients, so it should be used to decide which pages to fix."

**Rewritten (after):**
In one setting (features from one month, label from the next, pages with at least 500 impressions in both months, clients held out of training), the Random Forest's measured ROC-AUC averaged 0.829 vs 0.820 for the Week 4 rule under a client-grouped split, and 0.831 vs 0.825 when trained on February→March and tested on March→April. The gain ranged from +0.003 to +0.016 in the first design and from -0.001 to +0.011 in the second, so the difference is small and not consistent in the forward-in-time design. The forest was clearer at the top of the ranking (precision among the top 500 pages of about 0.93 to 0.94 vs about 0.875). This is a directional, decision-support result for ordering pages for human review. It does not show that the model improves outcomes, that it will hold in other months, or that changing a flagged page will raise its clicks. Most of the signal comes from March CTR relative to the position-tier median, which is also what the rule uses.

**A second claim from Week 4 (before):**
"These pages need their title and meta description fixed."

**Rewritten (after):**
"These pages ranked in similar positions but earned fewer clicks than comparable pages in March. They are candidates for review. This data cannot say why they underperform, and some may recover without any change, as several of the false-positive examples above did in April."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.